## Section 1: Setup SparkSession & Load Data

In [140]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType

spark = SparkSession.builder \
    .appName("EcommerceCustomerAnalysis") \
    .master("local[*]") \
    .getOrCreate()
print("Spark Session is ready!")

Spark Session is ready!


In [141]:
# load dataset
raw_df = spark.read.csv("ecommerce_customer_data.csv", header=True, inferSchema=True)
print(f"Loaded dataset with {raw_df.count()} rows and {len(raw_df.columns)} columns.")

Loaded dataset with 10000 rows and 23 columns.


In [142]:
raw_df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- IncomeLevel: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- TotalPurchases: double (nullable = true)
 |-- AverageOrderValue: double (nullable = true)
 |-- CustomerLifetimeValue: double (nullable = true)
 |-- FavoriteCategory: string (nullable = true)
 |-- SecondFavoriteCategory: string (nullable = true)
 |-- EmailEngagementRate: double (nullable = true)
 |-- SocialMediaEngagementRate: double (nullable = true)
 |-- MobileAppUsage: string (nullable = true)
 |-- CustomerServiceInteractions: double (nullable = true)
 |-- AverageSatisfactionScore: double (nullable = true)
 |-- EmailConversionRate: double (nullable = true)
 |-- SocialMediaConversionRate: double (nullable = true)
 |-- SearchEngineConversionRate: double (nullable = true)
 |-- RepeatCustomer: string (nu

In [143]:
# check first few rows
raw_df.show(5, truncate=False)

+----------+----------------+----+-----------------+-----------+---------+-----------+--------------+------------------+---------------------+----------------+----------------------+-------------------+-------------------------+--------------+---------------------------+------------------------+-------------------+-------------------------+--------------------------+--------------+-------------+----------------+
|CustomerID|RegistrationDate|Age |Gender           |IncomeLevel|Country  |City       |TotalPurchases|AverageOrderValue |CustomerLifetimeValue|FavoriteCategory|SecondFavoriteCategory|EmailEngagementRate|SocialMediaEngagementRate|MobileAppUsage|CustomerServiceInteractions|AverageSatisfactionScore|EmailConversionRate|SocialMediaConversionRate|SearchEngineConversionRate|RepeatCustomer|PremiumMember|HasReturnedItems|
+----------+----------------+----+-----------------+-----------+---------+-----------+--------------+------------------+---------------------+----------------+---------

## Section 2: Data Cleaning

In [144]:
# 2a. Remove exact duplicates
before_cnt = raw_df.count()
df_no_dupes = raw_df.dropDuplicates()
after_cnt = df_no_dupes.count()
print(f"Removed {before_cnt - after_cnt} duplicate rows. Remaining: {after_cnt}")

Removed 0 duplicate rows. Remaining: 10000


In [145]:
# 2b. Convert string 'nan' values to actual nulls
string_cols = ["Gender", "IncomeLevel", "MobileAppUsage", "FavoriteCategory", "SecondFavoriteCategory"]
df_clean = df_no_dupes

for col_name in string_cols:
    df_clean = df_clean.withColumn(
        col_name,
        F.when(F.col(col_name) == "nan", None).otherwise(F.col(col_name))
    )
print("Converted all literal 'nan' strings to actual nulls.")

Converted all literal 'nan' strings to actual nulls.


In [146]:
# 2c. Remove rows where CustomerID is null
before_id_cnt = df_clean.count()
df_clean = df_clean.filter(F.col("CustomerID").isNotNull())
after_id_cnt = df_clean.count()
print(f"Dropped {before_id_cnt - after_id_cnt} rows with null CustomerID. Remaining: {after_id_cnt}")

Dropped 492 rows with null CustomerID. Remaining: 9508


In [147]:
# 2d. Check for invalid Age values (negative, zero, or > 100) and set them to null
print("Sample of invalid age entries:")
df_clean.filter((F.col("Age") <= 0) | (F.col("Age") > 100)).select("CustomerID", "Age").show(5)

df_clean = df_clean.withColumn(
    "Age",
    F.when((F.col("Age") < 1) | (F.col("Age") > 100), None).otherwise(F.col("Age"))
)
print("Invalid age values set to null.")

Sample of invalid age entries:
+----------+-----+
|CustomerID|  Age|
+----------+-----+
| CUST00206|  0.0|
| CUST08551| -1.0|
| CUST04220|-21.0|
| CUST04348| -1.0|
| CUST09304| -4.0|
+----------+-----+
only showing top 5 rows
Invalid age values set to null.


In [148]:
# 2e. Handling outliers in AverageOrderValue (AOV)
print("AverageOrderValue range and summary stats:")
df_clean.select(F.min("AverageOrderValue"), F.max("AverageOrderValue"), F.avg("AverageOrderValue")).show()

# Let's cap extreme values at 5000
df_clean = df_clean.withColumn(
    "AverageOrderValue",
    F.when(F.col("AverageOrderValue") > 5000, None).otherwise(F.col("AverageOrderValue"))
)
print("Outliers in AverageOrderValue (> 5000) set to null for imputation.")

AverageOrderValue range and summary stats:
+----------------------+----------------------+----------------------+
|min(AverageOrderValue)|max(AverageOrderValue)|avg(AverageOrderValue)|
+----------------------+----------------------+----------------------+
|    1.2352819027124582|    51810.123750316256|    191.16656692923237|
+----------------------+----------------------+----------------------+

Outliers in AverageOrderValue (> 5000) set to null for imputation.


In [149]:
# 2f. Impute missing/null values
# Impute numeric columns with column means
num_cols = ["Age", "TotalPurchases", "AverageOrderValue", "CustomerLifetimeValue", 
            "CustomerServiceInteractions", "AverageSatisfactionScore",
            "EmailEngagementRate", "SocialMediaEngagementRate", 
            "EmailConversionRate", "SocialMediaConversionRate", "SearchEngineConversionRate"]

mean_values = {}
for c in num_cols:
    mean_val = df_clean.select(F.avg(c)).first()[0]
    if mean_val is not None:
        mean_values[c] = round(mean_val, 2)
print("Calculated means for imputation:", mean_values)
df_clean = df_clean.fillna(mean_values)

# Impute categories with 'Unknown' and Yes/No flags with 'No'
cat_cols = ["Gender", "IncomeLevel", "MobileAppUsage", "FavoriteCategory", "SecondFavoriteCategory", "Country", "City"]
df_clean = df_clean.fillna({c: "Unknown" for c in cat_cols})
df_clean = df_clean.fillna({c: "No" for c in ["RepeatCustomer", "PremiumMember", "HasReturnedItems"]})

print("Imputation complete. Verifying null count:")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()

Calculated means for imputation: {'Age': 35.03, 'TotalPurchases': 5.05, 'AverageOrderValue': 105.61, 'CustomerLifetimeValue': 685.07, 'CustomerServiceInteractions': 2.0, 'AverageSatisfactionScore': 6.96, 'EmailEngagementRate': 0.28, 'SocialMediaEngagementRate': 0.29, 'EmailConversionRate': 0.2, 'SocialMediaConversionRate': 0.2, 'SearchEngineConversionRate': 0.2}
Imputation complete. Verifying null count:
+----------+----------------+---+------+-----------+-------+----+--------------+-----------------+---------------------+----------------+----------------------+-------------------+-------------------------+--------------+---------------------------+------------------------+-------------------+-------------------------+--------------------------+--------------+-------------+----------------+
|CustomerID|RegistrationDate|Age|Gender|IncomeLevel|Country|City|TotalPurchases|AverageOrderValue|CustomerLifetimeValue|FavoriteCategory|SecondFavoriteCategory|EmailEngagementRate|SocialMediaEngagem

## Section 3: Schema Modifications (Renaming & Casting)

In [150]:
# Cast floating representation of counts to IntegerType
df_modified = df_clean \
    .withColumn("Age", F.col("Age").cast(IntegerType())) \
    .withColumn("TotalPurchases", F.col("TotalPurchases").cast(IntegerType())) \
    .withColumn("CustomerServiceInteractions", F.col("CustomerServiceInteractions").cast(IntegerType()))

# Parse date string
df_modified = df_modified.withColumn("RegistrationDate", F.to_date("RegistrationDate", "yyyy-MM-dd"))

# Rename long column names for easier querying
df_modified = df_modified \
    .withColumnRenamed("AverageOrderValue", "AvgOrderValue") \
    .withColumnRenamed("CustomerLifetimeValue", "CLV") \
    .withColumnRenamed("AverageSatisfactionScore", "AvgSatisfaction") \
    .withColumnRenamed("CustomerServiceInteractions", "ServiceInteractions") \
    .withColumnRenamed("EmailEngagementRate", "EmailEngagement") \
    .withColumnRenamed("SocialMediaEngagementRate", "SocialEngagement")

df_modified.printSchema()
df_modified.select("CustomerID", "Age", "RegistrationDate", "AvgOrderValue", "CLV").show(5)

root
 |-- CustomerID: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = false)
 |-- IncomeLevel: string (nullable = false)
 |-- Country: string (nullable = false)
 |-- City: string (nullable = false)
 |-- TotalPurchases: integer (nullable = true)
 |-- AvgOrderValue: double (nullable = false)
 |-- CLV: double (nullable = false)
 |-- FavoriteCategory: string (nullable = false)
 |-- SecondFavoriteCategory: string (nullable = false)
 |-- EmailEngagement: double (nullable = false)
 |-- SocialEngagement: double (nullable = false)
 |-- MobileAppUsage: string (nullable = false)
 |-- ServiceInteractions: integer (nullable = true)
 |-- AvgSatisfaction: double (nullable = false)
 |-- EmailConversionRate: double (nullable = false)
 |-- SocialMediaConversionRate: double (nullable = false)
 |-- SearchEngineConversionRate: double (nullable = false)
 |-- RepeatCustomer: string (nullable = false)
 |-- PremiumMember

## Section 4: Filtering Conditions

In [151]:
# 4a. Age filter (18 to 65)
adults = df_modified.filter((F.col("Age") >= 18) & (F.col("Age") <= 65))
print(f"Adults aged 18-65 count: {adults.count()}")

Adults aged 18-65 count: 8247


In [152]:
# 4b. Favorite category filter (Electronics fans)
electronics_fans = df_modified.filter(F.col("FavoriteCategory") == "Electronics")
print(f"Electronics lovers count: {electronics_fans.count()}")

Electronics lovers count: 1004


In [153]:
# 4c. USA customers
usa_customers = df_modified.filter(F.col("Country") == "USA")
print(f"USA customers count: {usa_customers.count()}")

USA customers count: 1130


In [154]:
# 4d. Compound Filter: Premium, High Income, High Satisfaction
premium_hv = df_modified.filter(
    (F.col("PremiumMember") == "Yes") &
    (F.col("IncomeLevel").isin("High", "Very High")) &
    (F.col("AvgSatisfaction") > 7.0)
)
print(f"Premium High-Value customers: {premium_hv.count()}")
premium_hv.select("CustomerID", "Country", "IncomeLevel", "CLV", "AvgSatisfaction").show(5)

Premium High-Value customers: 337
+----------+---------+-----------+------------------+-----------------+
|CustomerID|  Country|IncomeLevel|               CLV|  AvgSatisfaction|
+----------+---------+-----------+------------------+-----------------+
| CUST00954|    Other|       High| 309.7273978002596|9.454826919286411|
| CUST05619|       UK|       High|1225.8467742903233|8.077816489657863|
| CUST01106|Australia|  Very High| 194.4107526699019| 8.87109701384956|
| CUST03161|    Japan|  Very High|165.74032547816512|7.325486860666955|
| CUST09187|    Japan|  Very High|  579.295352391752|             10.0|
+----------+---------+-----------+------------------+-----------------+
only showing top 5 rows


## Section 5: Aggregation Functions

In [155]:
# 5a. Overall aggregations across the entire dataset
df_modified.agg(
    F.count("CustomerID").alias("total_customers"),
    F.sum("TotalPurchases").alias("total_purchases"),
    F.avg("AvgOrderValue").alias("overall_avg_order"),
    F.min("CLV").alias("min_clv"),
    F.max("CLV").alias("max_clv")
).show()

+---------------+---------------+-----------------+-----------------+-----------------+
|total_customers|total_purchases|overall_avg_order|          min_clv|          max_clv|
+---------------+---------------+-----------------+-----------------+-----------------+
|           9508|          47987|105.6124120563214|-9331.07701069297|420810.8156406275|
+---------------+---------------+-----------------+-----------------+-----------------+



In [156]:
# 5b. Calculated Revenue metric (AvgOrderValue * TotalPurchases) for top customers
df_rev = df_modified.withColumn(
    "EstRevenue", 
    F.round(F.col("AvgOrderValue") * F.col("TotalPurchases"), 2)
)
df_rev.select("CustomerID", "TotalPurchases", "AvgOrderValue", "EstRevenue")\
    .orderBy(F.desc("EstRevenue"))\
    .show(10)

+----------+--------------+------------------+----------+
|CustomerID|TotalPurchases|     AvgOrderValue|EstRevenue|
+----------+--------------+------------------+----------+
| CUST06568|            12|4423.9634894865585|  53087.56|
| CUST02971|            12| 4290.805579927638|  51489.67|
| CUST01639|            13| 3885.742309213363|  50514.65|
| CUST00066|            11| 4294.256403002562|  47236.82|
| CUST00415|            10|3432.6885542831806|  34326.89|
| CUST02854|             7| 4847.766139971882|  33934.36|
| CUST00275|             9| 3679.256101039663|   33113.3|
| CUST06447|            11|2850.1158670539053|  31351.27|
| CUST08139|             7| 4430.532878158056|  31013.73|
| CUST08035|             7| 4258.126383074894|  29806.88|
+----------+--------------+------------------+----------+
only showing top 10 rows


## Section 6: GroupBy & Having-like Conditions

In [157]:
# 6a. Group by Country with satisfaction filter > 6.5
df_modified.groupBy("Country").agg(
    F.count("CustomerID").alias("num_customers"),
    F.round(F.avg("AvgSatisfaction"), 2).alias("avg_satisfaction"),
    F.round(F.sum("TotalPurchases"), 2).alias("total_purchases")
).filter(F.col("avg_satisfaction") > 6.5)\
 .orderBy(F.desc("avg_satisfaction"))\
 .show()

+---------+-------------+----------------+---------------+
|  Country|num_customers|avg_satisfaction|total_purchases|
+---------+-------------+----------------+---------------+
|    Other|         1125|            7.05|           5807|
|    Japan|         1162|            7.04|           5763|
|   France|         1165|            6.96|           5912|
|   Canada|         1105|            6.96|           5616|
|  Germany|         1090|            6.95|           5406|
|Australia|         1142|            6.95|           5658|
|       UK|         1122|            6.94|           5664|
|      USA|         1130|            6.88|           5784|
|  Unknown|          467|            6.87|           2377|
+---------+-------------+----------------+---------------+



In [158]:
# 6b. Group by IncomeLevel
df_modified.groupBy("IncomeLevel").agg(
    F.count("CustomerID").alias("count"),
    F.round(F.avg("CLV"), 2).alias("avg_clv"),
    F.round(F.avg("AvgSatisfaction"), 2).alias("avg_satisfaction")
).orderBy(F.desc("avg_clv"))\
 .show()

+-----------+-----+-------+----------------+
|IncomeLevel|count|avg_clv|avg_satisfaction|
+-----------+-----+-------+----------------+
|  Very High| 1870| 805.03|            6.94|
|    Unknown| 2387| 758.45|            6.95|
|          L|    1| 732.46|            5.19|
|     Medium| 1814| 712.68|            6.98|
|        Low| 1733| 558.63|            6.98|
|       High| 1702| 549.87|            6.95|
|          H|    1| 320.75|             4.4|
+-----------+-----+-------+----------------+



## Section 7: Wide Transformations & Shuffling

In [159]:
# Explain physical plan showing execution stages
grouped_query = df_modified.groupBy("Country").agg(F.avg("CLV").alias("avg_clv"))
grouped_query.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Country#13612], functions=[avg(CLV#13774)])
   +- Exchange hashpartitioning(Country#13612, 200), ENSURE_REQUIREMENTS, [plan_id=26519]
      +- HashAggregate(keys=[Country#13612], functions=[partial_avg(CLV#13774)])
         +- HashAggregate(keys=[AverageOrderValue#12805, SecondFavoriteCategory#12808, TotalPurchases#12804, IncomeLevel#12801, SocialMediaEngagementRate#12810, HasReturnedItems#12819, AverageSatisfactionScore#12813, EmailConversionRate#12814, Gender#12800, SocialMediaConversionRate#12815, FavoriteCategory#12807, PremiumMember#12818, RepeatCustomer#12817, EmailEngagementRate#12809, Country#12802, CustomerID#12797, Age#12799, MobileAppUsage#12811, City#12803, RegistrationDate#12798, CustomerLifetimeValue#12806, CustomerServiceInteractions#12812, SearchEngineConversionRate#12816], functions=[])
            +- Exchange hashpartitioning(AverageOrderValue#12805, SecondFavoriteCategory#12808, TotalPurch

In [160]:
# Check default partition count
print(f"Initial partition count: {df_modified.rdd.getNumPartitions()}")

# Repartitioning by a key column (forces a full wide transformation shuffle)
repartitioned_df = df_modified.repartition(4, "Country")
print(f"Partition count after repartition: {repartitioned_df.rdd.getNumPartitions()}")

# Coalesce (narrow transformation - decreases partitions without a full shuffle)
coalesced_df = repartitioned_df.coalesce(2)
print(f"Partition count after coalesce: {coalesced_df.rdd.getNumPartitions()}")

Initial partition count: 1
Partition count after repartition: 4
Partition count after coalesce: 2


## Section 8: Complete Data Processing Pipeline

Here we execute a clean pipeline starting from the raw CSV data, completing all processes in order, and writing output as Parquet (for the full clean dataset) and CSV (for small aggregated reports).

In [161]:
def run_full_pipeline(input_path, output_dir):
    print("[1/5] Loading raw data...")
    df = spark.read.csv(input_path, header=True, inferSchema=True)
    
    print("[2/5] Cleaning data and handling nulls...")
    # converts string 'nan' values
    for col in ["Gender", "IncomeLevel", "MobileAppUsage", "FavoriteCategory", "SecondFavoriteCategory"]:
        df = df.withColumn(col, F.when(F.col(col) == "nan", None).otherwise(F.col(col)))
        
    df = df.dropDuplicates().filter(F.col("CustomerID").isNotNull())
    
    # fix anomalies
    df = df.withColumn("Age", F.when((F.col("Age") < 1) | (F.col("Age") > 100), None).otherwise(F.col("Age")))
    df = df.withColumn("AverageOrderValue", F.when(F.col("AverageOrderValue") > 5000, None).otherwise(F.col("AverageOrderValue")))
    
    # impute numeric columns
    num_impute = {}
    for c in ["Age", "TotalPurchases", "AverageOrderValue", "CustomerLifetimeValue", "AverageSatisfactionScore"]:
        val = df.select(F.avg(c)).first()[0]
        if val is not None:
            num_impute[c] = round(val, 2)
    df = df.fillna(num_impute)
    
    # impute categorical/binary columns
    df = df.fillna("Unknown", subset=["Gender", "IncomeLevel", "FavoriteCategory", "Country", "City"])
    df = df.fillna("No", subset=["RepeatCustomer", "PremiumMember", "HasReturnedItems"])
    
    print("[3/5] Transforming schema & casting...")
    df = df \
        .withColumn("Age", F.col("Age").cast(IntegerType())) \
        .withColumn("TotalPurchases", F.col("TotalPurchases").cast(IntegerType())) \
        .withColumn("RegistrationDate", F.to_date("RegistrationDate", "yyyy-MM-dd")) \
        .withColumn("EstRevenue", F.round(F.col("AverageOrderValue") * F.col("TotalPurchases"), 2))
        
    print("[4/5] Filtering (keeping ages 18-65)...")
    df_final = df.filter((F.col("Age") >= 18) & (F.col("Age") <= 65))
    
    print("[5/5] Creating aggregations and writing outputs...")
    #saving datasets into parquest and csv respectively
    df_final.write.mode("overwrite").parquet(os.path.join(output_dir, "cleaned_data"))
    
    cat_report = df_final.groupBy("FavoriteCategory").agg(
        F.count("CustomerID").alias("customers"),
        F.round(F.avg("AverageOrderValue"), 2).alias("avg_order_value"),
        F.round(F.sum("EstRevenue"), 2).alias("total_est_revenue")
    ).orderBy(F.desc("total_est_revenue"))
    cat_report.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(output_dir, "category_summary"))
    
    # save country report to CSV
    country_report = df_final.groupBy("Country").agg(
        F.count("CustomerID").alias("customers"),
        F.round(F.avg("CustomerLifetimeValue"), 2).alias("avg_clv"),
        F.round(F.sum("EstRevenue"), 2).alias("total_est_revenue")
    ).orderBy(F.desc("total_est_revenue"))
    country_report.coalesce(1).write.mode("overwrite").option("header", True).csv(os.path.join(output_dir, "country_summary"))
    
    print("Pipeline complete! Output saved under output/")
run_full_pipeline("ecommerce_customer_data.csv", "output")

[1/5] Loading raw data...


[2/5] Cleaning data and handling nulls...
[3/5] Transforming schema & casting...
[4/5] Filtering (keeping ages 18-65)...
[5/5] Creating aggregations and writing outputs...
Pipeline complete! Output saved under output/


In [162]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
